# GIAI ĐOẠN 4A: TIỀN XỬ LÝ SÂU & TÌM TỪ DỪNG MANG TÍN HIỆU CẢM XÚC

## 1. Phương pháp luận & Ý nghĩa bài toán
- **Vấn đề cốt lõi**:
  + Các kho từ dừng chung (`tu_dung.txt`) chứa rất nhiều từ có trong từ điển tiếng Việt nhưng lại **mang nặng sắc thái cảm xúc** trong mua sắm (ví dụ: *không, chẳng, chưa, quá, rất, tốt, tệ, hài lòng...*).
  + Nếu ta chưa có sẵn danh sách giữ lại (`giu_lai.txt`), làm sao nhóm biết được từ nào cần giữ, từ nào nên xóa một cách khách quan?
- **Giải pháp: Phân tích độ lệch lớp (Class Disproportion Analysis)**:
  + Nhóm dùng chính tập **Train** để kiểm tra: Khi một từ dừng $w$ xuất hiện trong câu, nó có làm thiên lệch xác suất rơi vào lớp Tiêu cực hay Tích cực không?
    $$\text{Độ lệch}(c) = \frac{P(c \mid \text{câu có từ } w)}{P(c)}$$
  + Nếu $\text{Độ lệch} \ge 1.5$ (tức là khi từ này xuất hiện, câu đó có xu hướng tiêu cực/tích cực gấp 1.5 lần bình thường) -> Từ đó mang tín hiệu cảm xúc mạnh -> **Đề xuất giữ lại** để tạo ra mức từ dừng lọc (`loc`).
- **So sánh 3 mức từ dừng trên Validation**:
  + `khong`: Giữ nguyên toàn bộ từ ngữ.
  + `nguyen`: Bỏ nguyên kho `tu_dung.txt` mà không giữ lại gì.
  + `loc`: Bỏ kho từ dừng nhưng **giữ lại các từ có độ lệch cao vừa tìm được**.

### Bước 1: Khai báo thư viện và đọc dữ liệu Train/Val

In [ ]:
import re
from collections import Counter
from pathlib import Path
import pandas as pd
from underthesea import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

# Thiết lập đường dẫn thư mục
SPLIT_DIR = Path("../data/splits")
RES_DIR   = Path("../data/resources")
OUT_DIR   = Path("../data/tien_xu_ly")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Đọc các tập dữ liệu đã chia từ Giai đoạn 3
df_train = pd.read_csv(SPLIT_DIR / "train.csv")
df_val   = pd.read_csv(SPLIT_DIR / "validation.csv")
df_test  = pd.read_csv(SPLIT_DIR / "test.csv")

print(f"-> Nạp thành công: Train={len(df_train)}, Val={len(df_val)}, Test={len(df_test)}")

# Nạp bảng Teencode và kho từ dừng chung ban đầu
df_teen = pd.read_csv(RES_DIR / "teencode.csv", dtype=str, encoding="utf-8-sig", keep_default_na=False)
BANG_TEENCODE = dict(zip(df_teen["viet_tat"].str.strip(), df_teen["day_du"].str.strip()))

def doc_danh_sach(path: Path) -> frozenset[str]:
    words = []
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#"):
                words.append(line.replace(" ", "_"))
    return frozenset(words)

KHO_TU_DUNG = doc_danh_sach(RES_DIR / "tu_dung.txt")
print(f"-> Đã nạp kho từ dừng gốc: {len(KHO_TU_DUNG):,} từ")

### Bước 2: Hàm tiền xử lý sâu cơ bản (Chữ thường, Teencode & Tách từ tiếng Việt)

In [ ]:
TU_RE        = re.compile(r"\w+")
SO_TRUOC_RE  = re.compile(r"\d\s*$")   # Bảo vệ đơn vị đo đứng sau số: '50 k', '5 kg'
CO_CHU_SO_RE = re.compile(r"\w")      # Token chứa ít nhất 1 ký tự chữ hoặc số

def tien_xu_ly_co_ban(cau: str) -> str:
    """
    Tiền xử lý văn bản tiếng Việt:
    - Chuyển chữ thường.
    - Dịch từ viết tắt/teencode (có bảo vệ đơn vị đo sau số).
    - Tách từ tiếng Việt bằng underthesea và loại bỏ dấu câu đơn lẻ.
    """
    if not isinstance(cau, str) or not cau.strip():
        return ""
    cau = cau.lower()
    
    # Dịch teencode
    def doi(m: re.Match) -> str:
        tu = m.group(0)
        if tu not in BANG_TEENCODE:
            return tu
        if SO_TRUOC_RE.search(cau[: m.start()]):
            return tu
        return BANG_TEENCODE[tu]
    cau = TU_RE.sub(doi, cau)
    
    # Tách từ tiếng Việt
    tokens = word_tokenize(cau, format="text").split()
    tokens = [t for t in tokens if CO_CHU_SO_RE.search(t)]
    return " ".join(tokens)

# Tách từ cho toàn bộ các tập
print("Đang tách từ cho các tập Train, Val, Test...")
df_train["tach_tu"] = df_train["text"].apply(tien_xu_ly_co_ban)
df_val["tach_tu"]   = df_val["text"].apply(tien_xu_ly_co_ban)
df_test["tach_tu"]  = df_test["text"].apply(tien_xu_ly_co_ban)
print("-> Hoàn tất tách từ cơ bản.")

### Bước 3: Phân tích độ lệch lớp trên tập Train để tự động tìm các từ dừng mang cảm xúc
- *Giả sử chưa có sẵn `giu_lai.txt`, nhóm tính toán độ lệch từng từ dừng để chứng minh từ nào mang tín hiệu cảm xúc.*

In [ ]:
# Tính tỉ lệ lớp chung của tập Train
ti_le_chung = df_train["label"].value_counts(normalize=True).to_dict()
cac_lop = [-1, 0, 1]
ten_lop = {-1: "Tiêu cực", 0: "Trung tính", 1: "Tích cực"}

# Đếm số câu chứa từng từ dừng theo từng lớp
dem_tu = {k: Counter() for k in cac_lop}
for nhan, cau in zip(df_train["label"], df_train["tach_tu"]):
    words_in_cau = set(cau.split()) & KHO_TU_DUNG
    for w in words_in_cau:
        dem_tu[nhan][w] += 1

# Tính độ lệch cho từng từ dừng
MIN_SO_CAU = 3       # Từ xuất hiện ít nhất trong 3 câu ở tập Train demo (tăng lên 20-50 khi chạy full)
NGUONG_LECH = 1.3    # Xuất hiện thiên lệch gấp 1.3 lần tỉ lệ lớp chung

bang_do_lech = []
for w in set().union(*dem_tu.values()):
    so_cau_theo_lop = {k: dem_tu[k][w] for k in cac_lop}
    tong_cau = sum(so_cau_theo_lop.values())
    if tong_cau < MIN_SO_CAU:
        continue
        
    # Tính độ lệch: P(lớp c | từ w) / P(lớp c)
    do_lech = {k: (so_cau_theo_lop[k] / tong_cau) / ti_le_chung[k] for k in cac_lop}
    lop_lech_nhat = max(do_lech, key=do_lech.get)
    max_lech = do_lech[lop_lech_nhat]
    
    bang_do_lech.append({
        "Từ dừng": w,
        "Số câu xuất hiện": tong_cau,
        "Lớp lệch nhất": ten_lop[lop_lech_nhat],
        "Độ lệch lớn nhất": round(max_lech, 2),
        "Đề xuất giữ lại": max_lech >= NGUONG_LECH
    })

df_lech = pd.DataFrame(bang_do_lech).sort_values(by=["Đề xuất giữ lại", "Độ lệch lớn nhất"], ascending=False)
tu_de_xuat_giu = set(df_lech[df_lech["Đề xuất giữ lại"]]["Từ dừng"])

print("=== BẢNG PHÂN TÍCH ĐỘ LỆCH CỦA TỪ DỪNG TRONG TẬP TRAIN ===")
display(df_lech.head(10))
print(f"\n-> Tìm được {len(tu_de_xuat_giu)} từ dừng mang tín hiệu cảm xúc mạnh cần giữ lại: {list(tu_de_xuat_giu)[:15]}...")

### Bước 4: Thực nghiệm so sánh 3 mức từ dừng trên tập Validation

In [ ]:
# Định nghĩa hàm lọc từ dừng theo 3 mức dựa trên kết quả tìm được ở Bước 3
def bo_tu_dung(tach_tu_str: str, muc: str) -> str:
    tokens = tach_tu_str.split()
    if muc == "khong":
        return tach_tu_str
    if muc == "nguyen":
        return " ".join([t for t in tokens if t not in KHO_TU_DUNG])
    if muc == "loc":
        # Chỉ bỏ từ dừng thông thường, GIỮ LẠI từ có độ lệch cảm xúc cao vừa tìm được
        return " ".join([t for t in tokens if (t not in KHO_TU_DUNG) or (t in tu_de_xuat_giu)])
    raise ValueError(f"Mức không hợp lệ: {muc}")

# Thử nghiệm lần lượt cả 3 mức trên Validation
TEN_MUC = {
    "khong": "1. Không bỏ từ dừng",
    "nguyen": "2. Bỏ nguyên kho (mù quáng)",
    "loc": "3. Bỏ kho đã lọc (giữ từ cảm xúc tìm được)"
}

ket_qua_so_sanh = []
for muc in ["khong", "nguyen", "loc"]:
    x_tr = df_train["tach_tu"].apply(lambda s: bo_tu_dung(s, muc))
    x_va = df_val["tach_tu"].apply(lambda s: bo_tu_dung(s, muc))
    
    vec = TfidfVectorizer(tokenizer=lambda s: s.split(), token_pattern=None, min_df=1, sublinear_tf=True)
    X_tr_vec = vec.fit_transform(x_tr)
    X_va_vec = vec.transform(x_va)
    
    clf = LinearSVC(C=1.0, class_weight="balanced", max_iter=3000, random_state=42)
    clf.fit(X_tr_vec, df_train["label"])
    
    du_doan = clf.predict(X_va_vec)
    acc = accuracy_score(df_val["label"], du_doan) * 100
    f1  = f1_score(df_val["label"], du_doan, average="macro")
    
    ket_qua_so_sanh.append({
        "Mức": muc,
        "Chiến lược xử lý": TEN_MUC[muc],
        "Số từ vựng (Vocab)": f"{len(vec.vocabulary_):,}",
        "Độ dài TB (từ/câu)": round(x_tr.str.split().str.len().mean(), 2),
        "Accuracy (Val)": f"{acc:.2f}%",
        "Macro-F1 (Val)": round(f1, 4)
    })

df_kq = pd.DataFrame(ket_qua_so_sanh)
print("=== BẢNG SO SÁNH HIỆU NĂNG 3 MỨC TỪ DỪNG TRÊN TẬP VALIDATION ===")
display(df_kq)

best_muc = df_kq.loc[df_kq["Macro-F1 (Val)"].idxmax(), "Mức"]
print(f"\n-> KẾT LUẬN: Mức [{TEN_MUC[best_muc]}] đạt Macro-F1 cao nhất. Nhóm chính thức chọn mức này!")

### Bước 5: Áp dụng mức tối ưu nhất và lưu các tệp tiền xử lý

In [ ]:
# Áp dụng mức tối ưu cho cả 3 tập
df_train["van_ban"] = df_train["tach_tu"].apply(lambda s: bo_tu_dung(s, best_muc))
df_val["van_ban"]   = df_val["tach_tu"].apply(lambda s: bo_tu_dung(s, best_muc))
df_test["van_ban"]  = df_test["tach_tu"].apply(lambda s: bo_tu_dung(s, best_muc))

# Lưu ra thư mục data/tien_xu_ly/
df_train[["van_ban", "label"]].to_csv(OUT_DIR / "train.csv", index=False, encoding="utf-8-sig")
df_val[["van_ban", "label"]].to_csv(OUT_DIR / "validation.csv", index=False, encoding="utf-8-sig")
df_test[["van_ban", "label"]].to_csv(OUT_DIR / "test.csv", index=False, encoding="utf-8-sig")

print("="*65)
print("HOÀN THÀNH GIAI ĐOẠN 4A (TIỀN XỬ LÝ SÂU):")
print(f"- Đã lưu 3 tập dữ liệu tại: {OUT_DIR.resolve()}")
print("="*65)
display(df_train[["van_ban", "label"]].head(3))